# 02 — Baseline & Model Comparison

Цель: обучить baseline (Logistic Regression), сравнить несколько моделей и построить ансамбль.

Весь код моделирования вынесен в `src/modeling.py`; здесь мы вызываем его функции и анализируем результаты.

**Метрики:** F1-score (приоритет) и ROC-AUC.

## 0. Настройка окружения

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f'Корень проекта: {project_root}')

Корень проекта: c:\Users\User\Desktop\ML2


In [2]:
from src.modeling import (
    load_splits,
    train_baseline,
    train_all_models,
    train_ensemble,
    evaluate,
    save_model,
)

## 1. Загрузка данных

In [3]:
X_train, y_train, X_val, y_val, X_test, y_test = load_splits(project_root)

Train: (19528, 18), Val: (4185, 18), Test: (4185, 18)


## 2. Baseline: Logistic Regression

Обучаем на исходных признаках (без engineered), без подбора гиперпараметров.

In [4]:
baseline_model, baseline_metrics = train_baseline(X_train, y_train, X_val, y_val)
print(f"\nBaseline F1 (val): {baseline_metrics['f1']:.4f}")
print(f"Baseline ROC-AUC (val): {baseline_metrics['roc_auc']:.4f}")


Baseline: Logistic Regression
  F1-Score : 0.8723
  ROC-AUC  : 0.9216
              precision    recall  f1-score   support

           0       0.83      0.80      0.81      1734
           1       0.86      0.88      0.87      2451

    accuracy                           0.85      4185
   macro avg       0.85      0.84      0.84      4185
weighted avg       0.85      0.85      0.85      4185


Baseline F1 (val): 0.8723
Baseline ROC-AUC (val): 0.9216


## 3. Сравнение моделей

Обучаем RandomForest, XGBoost, LightGBM, GradientBoosting на полном наборе признаков (включая engineered).

In [5]:
pipelines, results_df = train_all_models(X_train, y_train, X_val, y_val)
display(results_df)


RandomForest
  F1-Score : 0.8694
  ROC-AUC  : 0.9151
              precision    recall  f1-score   support

           0       0.83      0.79      0.81      1734
           1       0.86      0.88      0.87      2451

    accuracy                           0.84      4185
   macro avg       0.84      0.84      0.84      4185
weighted avg       0.84      0.84      0.84      4185


XGBoost
  F1-Score : 0.8677
  ROC-AUC  : 0.9140
              precision    recall  f1-score   support

           0       0.82      0.80      0.81      1734
           1       0.86      0.87      0.87      2451

    accuracy                           0.84      4185
   macro avg       0.84      0.84      0.84      4185
weighted avg       0.84      0.84      0.84      4185



c:\Users\User\Desktop\ML2\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\User\Desktop\ML2\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



LightGBM
  F1-Score : 0.8736
  ROC-AUC  : 0.9220
              precision    recall  f1-score   support

           0       0.83      0.81      0.82      1734
           1       0.87      0.88      0.87      2451

    accuracy                           0.85      4185
   macro avg       0.85      0.84      0.85      4185
weighted avg       0.85      0.85      0.85      4185


GradientBoosting
  F1-Score : 0.8770
  ROC-AUC  : 0.9230
              precision    recall  f1-score   support

           0       0.84      0.80      0.82      1734
           1       0.86      0.89      0.88      2451

    accuracy                           0.85      4185
   macro avg       0.85      0.85      0.85      4185
weighted avg       0.85      0.85      0.85      4185


Сводная таблица результатов (Val):
                        f1   roc_auc
model                               
GradientBoosting  0.876954  0.922962
LightGBM          0.873610  0.921990
RandomForest      0.869425  0.915139
XGBoost          

,f1,roc_auc
model,,
GradientBoosting,0.876954,0.922962
LightGBM,0.873610,0.921990
RandomForest,0.869425,0.915139
XGBoost,0.867680,0.914049


## 4. Ансамбль (Soft Voting)

Объединяем все обученные модели с помощью Soft Voting.

In [6]:
ensemble, ensemble_metrics = train_ensemble(pipelines, X_train, y_train, X_val, y_val)
print(f"\nEnsemble F1 (val): {ensemble_metrics['f1']:.4f}")
print(f"Ensemble ROC-AUC (val): {ensemble_metrics['roc_auc']:.4f}")

c:\Users\User\Desktop\ML2\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\User\Desktop\ML2\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Ensemble (Soft Voting)
  F1-Score : 0.8753
  ROC-AUC  : 0.9224
              precision    recall  f1-score   support

           0       0.83      0.80      0.82      1734
           1       0.86      0.89      0.88      2451

    accuracy                           0.85      4185
   macro avg       0.85      0.84      0.85      4185
weighted avg       0.85      0.85      0.85      4185


Ensemble F1 (val): 0.8753
Ensemble ROC-AUC (val): 0.9224


## 5. Выбор лучшей модели и тестовая оценка

In [7]:
import pandas as pd

# Сводная таблица всех моделей
summary = results_df.copy()
summary.loc['Baseline (LR)'] = baseline_metrics
summary.loc['Ensemble'] = ensemble_metrics
summary = summary.sort_values('f1', ascending=False)

print('Сводная таблица (Val):')
display(summary)

# Выбираем лучшую модель по F1
best_name = summary['f1'].idxmax()
print(f'\nЛучшая модель по F1: {best_name}')

if best_name == 'Ensemble':
    best_model = ensemble
elif best_name == 'Baseline (LR)':
    best_model = baseline_model
else:
    best_model = pipelines[best_name]

Сводная таблица (Val):


,f1,roc_auc
model,,
GradientBoosting,0.876954,0.922962
Ensemble,0.875327,0.922393
LightGBM,0.873610,0.921990
Baseline (LR),0.872332,0.921632
RandomForest,0.869425,0.915139
XGBoost,0.867680,0.914049



Лучшая модель по F1: GradientBoosting


In [8]:
# Финальная оценка на тестовой выборке
print('--- Финальная оценка на TEST ---')
test_metrics = evaluate(best_model, X_test, y_test, label=f'Best: {best_name}')
print(f"Test F1: {test_metrics['f1']:.4f} | Test ROC-AUC: {test_metrics['roc_auc']:.4f}")

--- Финальная оценка на TEST ---

Best: GradientBoosting
  F1-Score : 0.8713
  ROC-AUC  : 0.9186
              precision    recall  f1-score   support

           0       0.83      0.79      0.81      1735
           1       0.86      0.88      0.87      2450

    accuracy                           0.85      4185
   macro avg       0.84      0.84      0.84      4185
weighted avg       0.85      0.85      0.85      4185

Test F1: 0.8713 | Test ROC-AUC: 0.9186


In [9]:
# Сохраняем лучшую модель
safe_name = best_name.replace(' ', '_').replace('(', '').replace(')', '')
save_model(best_model, safe_name, project_root)

Модель сохранена: c:\Users\User\Desktop\ML2\models\GradientBoosting.pkl


WindowsPath('c:/Users/User/Desktop/ML2/models/GradientBoosting.pkl')